### Residential reconstruction — IECC Zone 4B

Applies the already-proven best-practice pipeline (established and validated on the Coast zone across `weather_correction.ipynb`, `model_comparison.ipynb`, `feature_engineering.ipynb`) directly to zone **4B**, using the new statewide bus mapping. Does not re-derive the temporal-leakage fix, the weather-geography fix, the model comparison, or the feature ablation — those conclusions are settled and reused as-is:

- Population-weighted county weather blend (proven fix for the original PUMA-vs-county geography bug).
- Cyclical calendar features + lagged/rolling temperature (the feature set that won the ablation study).
- LightGBM (the model-comparison winner).
- Blocked seasonal 4-fold CV for the honest R2 estimate; final production models trained on the full year.

Depends on `00_zone_mapping.ipynb` having been run first (reads `county_to_iecc_zone.csv`, `bus_zone_master.csv`, `tract_bus_weights.csv`, `zone_county_weather_weights.csv`).

In [1]:
# env1 3.11.13 (repo default)
import pandas as pd
import numpy as np
from pathlib import Path
import requests
import joblib
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score, mean_squared_error

ZONE = "4B"
ZONE_PREFIX = "4b"

exports = Path("../../loadcurves/exports")
statewide_dir = Path("..")
weather_dir = statewide_dir / "weather_by_county"
weather_dir.mkdir(exist_ok=True)
models_dir = statewide_dir / "models" / ZONE
models_dir.mkdir(parents=True, exist_ok=True)
metrics_dir = statewide_dir / "metrics"
metrics_dir.mkdir(exist_ok=True)
exports_out_dir = statewide_dir / "exports"
exports_out_dir.mkdir(exist_ok=True)

# Authoritative bus table (every Texas 7k bus, with its IECC zone, county and case load)
# and the area-split tract -> bus weights. Both built by 00_zone_mapping.ipynb from the two
# ArcGIS exports; they replace the old single-nearest-bus `bus_to_zone_full.csv`.
bus_master = pd.read_csv(statewide_dir / "bus_zone_master.csv", dtype={"COUNTYFP": str})
bus_master["COUNTYFP"] = bus_master["COUNTYFP"].str.zfill(3)
zone_buses = bus_master[bus_master["iecc_zone"] == ZONE].copy()

tract_bus_w = pd.read_csv(
    statewide_dir / "tract_bus_weights.csv",
    dtype={"GEOID": str, "COUNTYFP": str, "TRACTCE": str},
)
zone_tract_bus = tract_bus_w[tract_bus_w["iecc_zone"] == ZONE].copy()

weights_df = pd.read_csv(statewide_dir / "zone_county_weather_weights.csv", dtype={"county_fips": str})
zone_weights = weights_df[weights_df["iecc_zone"] == ZONE].set_index("county_fips")["weight"]

n_load_buses = (zone_buses["Load_MW"] > 0).sum()
print(f"Zone {ZONE}: {len(zone_buses)} buses ({n_load_buses} carrying case load, "
      f"{zone_buses['Load_MW'].sum():,.0f} MW in the Texas 7k case), {len(zone_weights)} counties")
print(f"  tract->bus area weights: {zone_tract_bus['GEOID'].nunique()} tracts, "
      f"{len(zone_tract_bus)} tract-bus pairs")


Zone 4B: 124 buses (64 carrying case load, 1,221 MW in the Texas 7k case), 33 counties
  tract->bus area weights: 152 tracts, 439 tract-bus pairs


#### 1. Population-weighted county weather blend

In [2]:
weather_cols = [
    "Dry Bulb Temperature [\u00b0C]", "Relative Humidity [%]", "Wind Speed [m/s]",
    "Global Horizontal Radiation [W/m2]",
]

base_url = (
    "https://oedi-data-lake.s3.amazonaws.com/nrel-pds-building-stock/"
    "end-use-load-profiles-for-us-building-stock/2025/resstock_amy2018_release_1/"
    "weather/state=TX/"
)

county_weather = {}
for county_fips in zone_weights.index:
    gid = f"G48{int(county_fips) * 10:05d}"
    local_path = weather_dir / f"{gid}_2018.csv"
    if not local_path.exists():
        r = requests.get(f"{base_url}{gid}_2018.csv", timeout=60)
        r.raise_for_status()
        local_path.write_bytes(r.content)
    df = pd.read_csv(local_path)
    df["date_time"] = pd.to_datetime(df["date_time"])
    county_weather[county_fips] = df.sort_values("date_time").reset_index(drop=True).set_index("date_time")[weather_cols]

print(f"Downloaded/loaded weather for {len(county_weather)}/{len(zone_weights)} zone {ZONE} counties.")

lengths = {k: len(v) for k, v in county_weather.items()}
assert len(set(lengths.values())) == 1, f"Row count mismatch across counties: {lengths}"

blended = sum(county_weather[c] * zone_weights[c] for c in county_weather).reset_index()
print(f"Blended weather shape: {blended.shape}")


Downloaded/loaded weather for 33/33 zone 4B counties.
Blended weather shape: (8760, 5)


#### 2. Feature engineering (cyclical calendar + lagged/rolling temperature — the winning set)

In [3]:
# LightGBM/XGBoost reject "[" "]" in feature names, so sanitize the weather column names
safe_names = {
    "Dry Bulb Temperature [\u00b0C]": "Temp_C",
    "Relative Humidity [%]": "RH_pct",
    "Wind Speed [m/s]": "WindSpeed_mps",
    "Global Horizontal Radiation [W/m2]": "GHI_Wm2",
}
w = blended.rename(columns=safe_names).copy()
weather_cols_safe = [safe_names[c] for c in weather_cols]

w["hour"] = w["date_time"].dt.hour
w["month"] = w["date_time"].dt.month
w["day_of_week"] = w["date_time"].dt.dayofweek
w["month_sin"] = np.sin(2 * np.pi * w["month"] / 12)
w["month_cos"] = np.cos(2 * np.pi * w["month"] / 12)
w["hour_sin"] = np.sin(2 * np.pi * w["hour"] / 24)
w["hour_cos"] = np.cos(2 * np.pi * w["hour"] / 24)
w["dow_sin"] = np.sin(2 * np.pi * w["day_of_week"] / 7)
w["dow_cos"] = np.cos(2 * np.pi * w["day_of_week"] / 7)

w["Temp_lag1h"] = w["Temp_C"].shift(1).bfill()
w["Temp_lag3h"] = w["Temp_C"].shift(3).bfill()
w["Temp_roll24h"] = w["Temp_C"].rolling(window=24, min_periods=1).mean()

feature_cols = weather_cols_safe + [
    "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
    "Temp_lag1h", "Temp_lag3h", "Temp_roll24h",
]

weather_clean = w
X = w[feature_cols]
print(f"X shape: {X.shape}")

X shape: (8760, 13)


#### 3. Load & normalize the 5 ResStock archetype profiles for zone 4B

In [4]:
def aggregate_to_hourly(df_15min, target_col, unit_col):
    df = df_15min.copy()
    time_col = "date_time" if "date_time" in df.columns else "timestamp"
    if time_col in df.columns:
        df[time_col] = pd.to_datetime(df[time_col])
        df = df.set_index(time_col)
    else:
        df.index = pd.date_range(start="2018-01-01 00:15:00", periods=len(df), freq="15min")

    unit_count = df[unit_col].iloc[0] if unit_col in df.columns else 1.0
    if unit_count == 0:
        unit_count = 1.0

    df_hourly = df[[target_col]].resample("h", label="right", closed="right").sum()
    if not df_hourly.empty and df_hourly.index[0].hour == 0 and df_hourly.index[0].minute == 0:
        df_hourly = df_hourly.iloc[1:]

    df_hourly[target_col] = df_hourly[target_col] / unit_count
    return df_hourly


RES_TARGET_COL = "out.electricity.total.energy_consumption..kwh"

res_building_types = [
    "mobile_home", "single-family_detached", "single-family_attached",
    "multi-family_with_2_-_4_units", "multi-family_with_5plus_units",
]

archetype_base_url = (
    "https://oedi-data-lake.s3.amazonaws.com/nrel-pds-building-stock/"
    "end-use-load-profiles-for-us-building-stock/2025/resstock_amy2018_release_1/"
    "timeseries_aggregates/by_ashrae_iecc_climate_zone_2004/upgrade=0/"
    f"ashrae_iecc_climate_zone_2004={ZONE}/"
)

hourly_profiles = {}
for b in res_building_types:
    fname = f"up00-{ZONE_PREFIX}-{b}.csv"
    local_path = exports / fname
    if not local_path.exists():
        r = requests.get(archetype_base_url + fname, timeout=120)
        r.raise_for_status()
        local_path.write_bytes(r.content)
    raw = pd.read_csv(local_path)
    hourly_profiles[b] = aggregate_to_hourly(raw, RES_TARGET_COL, "units_represented")

for name, d in hourly_profiles.items():
    if len(d) != len(X):
        print(f"  !! length mismatch for {name}: {len(d)} rows vs X {len(X)} rows")
print(f"Loaded {len(hourly_profiles)} residential profiles for zone {ZONE}.")


Loaded 5 residential profiles for zone 4B.


#### 4. Honest evaluation: blocked seasonal 4-fold CV (LightGBM)

In [5]:
SEASON_MONTHS = {"winter": [12, 1, 2], "spring": [3, 4, 5], "summer": [6, 7, 8], "fall": [9, 10, 11]}


def seasonal_blocked_cv(X, y, months_series):
    months_arr = months_series.values
    scores = {}
    for season, months in SEASON_MONTHS.items():
        test_mask = np.isin(months_arr, months)
        X_train, X_test = X[~test_mask], X[test_mask]
        y_train, y_test = y[~test_mask], y[test_mask]
        model = LGBMRegressor(n_estimators=100, max_depth=-1, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        scores[season] = r2_score(y_test, preds)
    return scores


cv_results = []
for b_type, df_hourly in hourly_profiles.items():
    y = df_hourly[RES_TARGET_COL].values
    season_scores = seasonal_blocked_cv(X, y, weather_clean["month"])
    mean_r2 = np.mean(list(season_scores.values()))
    cv_results.append({"Zone": ZONE, "Building_Type": b_type, "Mean_R2": mean_r2, **season_scores})
    print(f" -> {b_type:35s} mean R2={mean_r2:.4f}  " + "  ".join(f"{s}={v:.3f}" for s, v in season_scores.items()))

cv_df = pd.DataFrame(cv_results)
cv_df.to_csv(metrics_dir / f"seasonal_cv_{ZONE}.csv", index=False)


 -> mobile_home                         mean R2=0.7046  winter=0.587  spring=0.675  summer=0.898  fall=0.659


 -> single-family_detached              mean R2=0.3276  winter=0.048  spring=0.102  summer=0.758  fall=0.403


 -> single-family_attached              mean R2=0.3834  winter=0.202  spring=0.174  summer=0.737  fall=0.421


 -> multi-family_with_2_-_4_units       mean R2=0.1485  winter=0.086  spring=-0.385  summer=0.897  fall=-0.004


 -> multi-family_with_5plus_units       mean R2=0.6097  winter=0.447  spring=0.553  summer=0.913  fall=0.526


#### 5. Train final production models on the full year

In [6]:
for b_type, df_hourly in hourly_profiles.items():
    y = df_hourly[RES_TARGET_COL].values
    model = LGBMRegressor(n_estimators=100, max_depth=-1, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(X, y)
    safe_name = b_type.replace(" ", "_").replace("/", "-")
    joblib.dump(model, models_dir / f"model_{safe_name}.joblib")

print(f"5 final models saved to {models_dir}")


5 final models saved to ../models/4B


#### 6. Building counts per bus (zone 4B) and bus-level reconstruction

In [7]:
nrel_mapping = {
    "single-family_detached": ["B25024_002E"],
    "single-family_attached": ["B25024_003E"],
    "mobile_home": ["B25024_010E"],
    "multi-family_with_2_-_4_units": ["B25024_004E", "B25024_005E"],
    "multi-family_with_5plus_units": ["B25024_006E", "B25024_007E", "B25024_008E", "B25024_009E"],
}

# 2020-tract-vintage census (matches the tract boundaries of the statewide ArcGIS export;
# the original census_b25024_state_48_tracts.csv is 2010-vintage, correct for the old
# Coast-only file but mismatched with this statewide file — see 00_zone_mapping.ipynb)
census = pd.read_csv(statewide_dir / "census_b25024_state_48_tracts_2020vintage.csv", dtype={"state": str, "county": str, "tract": str})
census["GEOID"] = census["state"].str.zfill(2) + census["county"].str.zfill(3) + census["tract"].str.zfill(6)
census_cols = [c for c in census.columns if c.startswith("B25024")]
for c in census_cols:
    census[c] = pd.to_numeric(census[c], errors="coerce").fillna(0)

for b_type, cols in nrel_mapping.items():
    census[b_type] = census[cols].sum(axis=1)

group_cols = list(nrel_mapping.keys())

# Area-split allocation: each tract was subdivided into 50 equal-area parts in ArcGIS and each
# part matched to its nearest bus, so weight = (parts won by that bus) / 50. A tract's housing
# is therefore spread over every bus that serves part of it instead of going entirely to the
# single nearest one — this is the spatial-assignment refinement listed in the paper (§IV-F).
alloc = zone_tract_bus.merge(census[["GEOID"] + group_cols], on="GEOID", how="left")
n_missing = alloc[group_cols].isna().all(axis=1).sum()
alloc[group_cols] = alloc[group_cols].fillna(0.0).mul(alloc["weight"], axis=0)

final_bus_counts = (
    alloc.groupby("Bus_Number")[group_cols].sum()
    .reindex(zone_buses["Bus_Number"].values).fillna(0.0)
    .rename_axis("Bus_Number").reset_index()
)
final_bus_counts["Bus_Number"] = final_bus_counts["Bus_Number"].astype(str)

n_zero = (final_bus_counts[group_cols].sum(axis=1) == 0).sum()
n_zero_load = ((final_bus_counts[group_cols].sum(axis=1) == 0).values & (zone_buses["Load_MW"] > 0).values).sum()
print(f"Zone {ZONE}: {len(final_bus_counts)} buses, {n_zero} with zero housing "
      f"({n_zero_load} of which carry case load); {n_missing} tract-bus pairs had no census row")
print(f"  housing units allocated: {final_bus_counts[group_cols].to_numpy().sum():,.0f}")
final_bus_counts.to_csv(exports_out_dir / f"bus_building_counts_{ZONE}.csv", index=False)


Zone 4B: 124 buses, 9 with zero housing (0 of which carry case load); 0 tract-bus pairs had no census row
  housing units allocated: 210,152


In [8]:
base_profiles = {}
for b_type in hourly_profiles:
    safe_name = b_type.replace(" ", "_").replace("/", "-")
    model = joblib.load(models_dir / f"model_{safe_name}.joblib")
    base_profiles[b_type] = model.predict(X)

base_profiles_df = pd.DataFrame(base_profiles, index=weather_clean["date_time"])

counts_matrix = final_bus_counts.set_index("Bus_Number")[base_profiles_df.columns].fillna(0)
bus_profiles_df = base_profiles_df.dot(counts_matrix.T)

out_path = exports_out_dir / f"residential_bus_load_profiles_8760_{ZONE}.csv"
bus_profiles_df.to_csv(out_path)
print(f"Zone {ZONE} residential bus profiles: {bus_profiles_df.shape} -> {out_path}")
print(f"Peak (sum of all buses): {(bus_profiles_df.sum(axis=1)/1000).max():,.1f} MW (uncalibrated)")


Zone 4B residential bus profiles: (8760, 124) -> ../exports/residential_bus_load_profiles_8760_4B.csv
Peak (sum of all buses): 594.2 MW (uncalibrated)


#### Summary — Zone 4B

20-line honest R2 table saved to `metrics/seasonal_cv_4B.csv`, 5 production models saved to `models/4B/`, bus-level residential profiles saved to `exports/residential_bus_load_profiles_8760_4B.csv`. Consumed by `statewide_summary.ipynb` for the cross-zone comparison.